<div align="center">

# Encode a file and pick up where you stopped

Turn a text file into vectors and continue after an interruption.

</div>

## When do you need this?

You have an export of reviews, articles, or support messages.
The file may be too large for one in-memory matrix, or the job may stop halfway through.
**Save completed work** so a restart can continue from the first missing row.

## The story in pictures

<picture>
<source media="(prefers-color-scheme: dark)" srcset="media/03-files-with-resume-dark.svg">
<img src="media/03-files-with-resume-light.svg" alt="A text file feeds small batches into fse and a local vector store. A restart skips rows already saved.">
</picture>

## 1. Prepare a real text file

We use 1,000 articles from the [AG News test split](https://huggingface.co/datasets/SetFit/ag_news) and POTION-base-8M.
**One line owns one row**, including an empty line.
The [shared setup](README.md#run-a-recipe) installs the pinned packages and explains the temporary workspace.

In [ ]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

In [ ]:
import fse

encoder = recipe.encoder()
texts = recipe.news(12)
print(texts[0])
print("Vector dimensions:", encoder.output_dim)

In [ ]:
texts = recipe.news(1000)
source = work / "articles.txt"
source.write_text("\n".join(t.replace("\n", " ") for t in texts) + "\n", encoding="utf-8")
store_path = work / "article-vectors"
print("Input articles:", len(texts))

## 2. Read small batches

This reader holds at most 200 lines at a time and rejects records larger than 64 KiB.
The `skip` argument tells it how many saved rows to pass over.
**Keep the input unchanged** between runs, so row numbers keep the same meaning.

In [ ]:
MAX_RECORD_BYTES = 64 * 1024


def line_batches(path, skip, size=200):
    batch = []
    with path.open("rb") as stream:
        index = 0
        while raw := stream.readline(MAX_RECORD_BYTES + 1):
            if len(raw) > MAX_RECORD_BYTES:
                raise ValueError(f"line {index} exceeds {MAX_RECORD_BYTES} bytes")
            text = raw.rstrip(b"\n").decode("utf-8")
            if index >= skip:
                batch.append(text)
                if len(batch) == size:
                    yield batch
                    batch = []
            index += 1
    if batch:
        yield batch

## 3. Continue from the saved row count

`ShardSink` writes groups of rows to disk, called **shards**.
Its `first_row` tells us how many rows the store already has.
A **durable** row has been saved so it survives a process crash.
During a run, `durable_rows` counts only newly durable rows, so add it to `first_row`.

In [ ]:
def resume(stop_after=None):
    with fse.ShardSink(store_path, encoder, shard_rows=200) as sink:
        start = sink.first_row  # existing store prefix, not this run's durable count
        stream = encoder.encode_batches(line_batches(source, start), sink=sink, workers=1)
        try:
            for number, batch in enumerate(stream, 1):
                assert batch.durable_rows is not None
                # Only this prefix can be acknowledged before close.
                confirmed = start + batch.durable_rows
                print("Durable rows:", confirmed)
                if stop_after is not None and number == stop_after:
                    break
        finally:
            stream.close()
    with fse.VectorStore.open(store_path) as store:
        return len(store)

## 4. Stop, restart, and repeat

The first call stops after two batches.
The second finishes the file, and the third proves that **rerunning adds no duplicate rows**.
The final check compares every stored vector with fresh encoder output.

In [ ]:
first = resume(stop_after=2)
second = resume()
third = resume()
assert first == 400 and second == third == 1000
with fse.VectorStore.open(store_path) as store:
    offset = 0
    for batch in line_batches(source, 0):
        actual = store.get_vectors(start=offset, stop=offset + len(batch))
        np.testing.assert_array_equal(actual, encoder.encode(batch, workers=1))
        offset += len(batch)
    assert offset == len(store)
print({"interrupted_rows": first, "resumed_rows": second, "rerun_rows": third})

## Result

The store grows from 400 rows to 1,000, then stays at 1,000 on the next run.
**Resume from the saved prefix**, the uninterrupted sequence of rows starting at zero.
This example stops cleanly, rather than cutting power, so it checks restart logic and row identity.

## Before using your own export

Use one writer and keep the vector store on a supported local filesystem.
Retain the input's identity, encoder fingerprint, and batch settings with a production job.
An encoder fingerprint identifies the model and its settings, and a changed input requires a new job.
The local store guarantee does not extend to SMB or NFS network storage.

## What next?

- [Watch a folder for new files](05-folder-listener.ipynb).
- [fse API and design](../docs/architecture/store.md)
- [Python file I/O](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)

Back to the [recipe index](README.md) or the [main README](../README.md#recipes-from-a-model-to-a-working-application).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [ ]:
recipe.close()